In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
%run /Workspace/fmcg_domain/databricks_fmcg_data_engineering/Setup/utilities

In [0]:
bronze_schema,silver_schema,gold_schema
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("datasource","customers","Data source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("datasource")

In [0]:
silver_gross_price_table = f"{catalog}.{silver_schema}.{data_source}"
silver_gross_price_table

In [0]:
gold_df = spark.sql(f"""
SELECT product_code, gross_price as price_inr, year
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY product_code
               ORDER BY month DESC
           ) AS rn
    FROM fmcg.silver.gross_price
) t
WHERE rn = 1;""" )

In [0]:
display(gold_df)

In [0]:
gold_df.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
            .option("delta.enableChangeDataFeed","true")\
                .saveAsTable("fmcg.gold.sb_dim_gross_price")

In [0]:
DeltaTable.forName(spark , "fmcg.gold.dim_gross_price").alias("target")\
    .merge(
        gold_df.alias("source"),
        "target.product_code = source.product_code"
    )\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()